# Synthetic claims journeys and time-to-event methods

This notebook demonstrates deterministic claims engineering and Kaplan–Meier calculations using a **handcrafted, schema-compatible fixture only**. The fixture is labeled `public_synthetic` to exercise the CMS DE-SynPUF 2008–2010 contract, but it is **not an extract of actual CMS source rows**. The method is for CMS synthetic-data demonstration only; results are **not representative of Medicare beneficiaries** and no estimate is generalized to Medicare.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ckd_intelligence.journeys.synpuf import (  # noqa: E402
    build_journey_output,
    validate_journey_order,
    validate_synpuf_fixture_manifest,
)
from ckd_intelligence.statistics.time_to_event import kaplan_meier  # noqa: E402

manifest = ROOT / 'data' / 'manifests' / 'synpuf-journeys-fixture.json'
fixture = ROOT / 'data' / 'fixtures' / 'synpuf_journeys.csv'
validate_synpuf_fixture_manifest(manifest, fixture)
claims = pd.read_csv(fixture)
output = build_journey_output(claims)
validate_journey_order(output.events)
assert set(output.events['evidence_type']) == {'public_synthetic'}
print({
    'evidence_type': 'public_synthetic',
    'fixture_kind': 'fixture_only',
    'source_rows_are_official': False,
    'source_documentation': 'CMS DE-SynPUF 2008-2010',
    'rows': len(claims),
})
print(output.rules.rule_text)

{'evidence_type': 'public_synthetic', 'fixture_kind': 'fixture_only', 'source_rows_are_official': False, 'source_documentation': 'CMS DE-SynPUF 2008-2010', 'rows': 6}
Index date is the first valid claim date per synthetic beneficiary. The target is the first diagnosis code beginning with one of the CKD ICD-9 prefixes ('585', '586', '588'). Events are sorted by event date, through date, and claim ID. Follow-up ends at the earlier of the global observation end and index plus the configured follow-up window. A target is persistent only when another CKD event occurs at least the configured 90-day gap later (the boundary is inclusive). People with no valid claim are excluded; those with no target are retained as right-censored in the persistence demonstration.


## Journey rules

The index date is the first valid claim date per synthetic beneficiary. The target is the first CKD ICD-9 signal. A persistence event requires a second CKD signal at least 90 days later, including the 90-day boundary. A person without a persistence event is right-censored at the earlier of the 365-day follow-up end or the CMS 2010 observation end. These are software-method rules, not clinical definitions.

In [2]:
event_summary = (
    output.events.groupby(
        ['event_kind', 'is_ckd_signal', 'is_target_event', 'is_persistence_event'],
        dropna=False,
    )
    .size()
    .rename('event_rows')
    .reset_index()
)
print('Identifier-free synthetic journey event summary:')
display(event_summary)
journey_summary = output.summaries.drop(columns=['synthetic_id'], errors='ignore')
print('Identifier-free synthetic journey summaries:')
display(journey_summary)

Identifier-free synthetic journey event summary:


,event_kind,is_ckd_signal,is_target_event,is_persistence_event,event_rows
0,follow_up,True,False,True,1
1,index,False,False,False,2
2,index,True,True,False,1
3,target,True,True,False,2


Identifier-free synthetic journey summaries:


,index_date,target_date,persistence_date,observation_end,duration_days,event_observed,censoring_reason,has_target,evidence_type
0,2009-01-01,2009-01-10,2009-04-10,2010-01-01,99,True,None,True,public_synthetic
1,2009-02-01,2009-02-01,None,2010-02-01,365,False,right_censored_at_follow_up_end,True,public_synthetic
2,2009-01-01,2009-05-01,None,2010-01-01,365,False,right_censored_at_follow_up_end,True,public_synthetic


In [3]:
km = kaplan_meier(output.summaries['duration_days'], output.summaries['event_observed'])
display(km.table)
assert km.table.equals(output.survival.drop(columns='evidence_type'))
print({
    'median_survival_days': km.median_survival,
    'confidence_level': km.confidence_level,
    'evidence_type': 'public_synthetic',
})
assert (km.table['survival'].diff().dropna() <= 0).all()
print(
    'Interpretation boundary: this Kaplan–Meier table describes only these CMS '
    'synthetic records and is not representative of Medicare beneficiaries.'
)

,time,at_risk,events,censored,survival,greenwood_variance,ci_low,ci_high
0,99.0,3,1,0,0.666667,0.074074,0.133232,1.0
1,365.0,2,0,2,0.666667,0.074074,0.133232,1.0


{'median_survival_days': None, 'confidence_level': 0.95, 'evidence_type': 'public_synthetic'}
Interpretation boundary: this Kaplan–Meier table describes only these CMS synthetic records and is not representative of Medicare beneficiaries.


## Limitations

CMS DE-SynPUF is a public synthetic sample. It does not support claims about real Medicare utilization, persistence, incidence, treatment, outcomes, or patient journeys. The notebook intentionally keeps this output separate from all public-observed NHANES, MEPS, Part D, and ClinicalTrials.gov analyses.